# Physics-informed fine-tuning of the PhysicsNeMo Stokes surrogate on a Kaggle GPU

Launcher for [physicsnemo-stokes-finetuning](https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

| Step | What runs | When |
|---|---|---|
| A | install, dataset download with SHA-256 check, seeded split, tests | always |
| B | CUDA smoke study (both stages, tiny) | always |
| A2 | restore the run of the previous session from `state-full.tar` (checksums of the archive and of every checkpoint) | `RESUME = True` |
| C0 | device checks: resume after an interruption, repeatability, checkpoint round trip, gradients | always |
| C | timing and memory probe: one epoch of the data-driven stage, 200 fine-tuning iterations | always |
| D | full study, `configs/full.yaml`: 500 epochs on 500 meshes, then 10000 fine-tuning iterations on each of 10 test meshes | `RUN_FULL = True` |

Before D the notebook prints the runtime projected from the probe. The configuration is **never reduced**. Step D runs with `--max-hours` set to what is left of `SESSION_BUDGET_HOURS` minus a margin: if the study cannot finish, training pauses at an epoch boundary (or fine-tuning between meshes) with a verified checkpoint, and the whole run directory is written to `state-full.tar` in the output. To continue, start a new version or kernel with this version's output attached as a kernel source and `RESUME = True`; step A2 restores and verifies the state and step D continues where it stopped. The state is packed after every session, also when the study fails.

There is no separate `RUN_OFFICIAL` step. `configs/official.yaml` holds NVIDIA's settings and fine-tunes test mesh 7 only, as the upstream script does; `configs/full.yaml` uses the same settings and seed for both stages and fine-tunes test meshes 0 to 9, so the upstream case is sample 7 of step D.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning.git"
REF = "9105b9539ac7ec09ba32cd2bb2ec7bc9b8ee4c77"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step D: only the value True starts it
CONFIG = "full"                               # configuration of step D (configs/<CONFIG>.yaml); its run directory is runs/<CONFIG>
STAGES = "train,predict,finetune,evaluate"    # stages of step D
RESUME = False                                # continuation session: attach the previous session's output (state-<CONFIG>.tar) as a kernel source
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-stokes-finetuning"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
assert tree_is_clean(), "the existing working tree is not clean"
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

## Step A: environment, dataset, tests

The requirements pin PhysicsNeMo to a commit of its main branch, which needs `torch>=2.13`; `pip` may replace the PyTorch build of the image. `torch_scatter` has no prebuilt wheel for that PyTorch and is compiled from source (CPU sources only; the sum aggregation of MeshGraphNet runs through `Tensor.scatter_add_` on the GPU). The versions are printed afterwards in a new process, and CUDA is checked again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh("FORCE_ONLY_CPU=1 pip install -q --no-build-isolation --no-binary torch-scatter torch-scatter==2.1.2", cwd=REPO)
# The Kaggle image preinstalls RAPIDS (cuml, cudf, cupy, rmm, ...) and numba. PhysicsNeMo imports cuml
# whenever it is installed; cudf then imports numba, which rejects the NumPy that requirements.txt
# installs (first launch, 2026-11-07: "Numba needs NumPy 2.2 or less. Got NumPy 2.5"). The validated
# environment (requirements.lock.txt) contains none of these packages, so they are removed from this
# session instead of downgrading NumPy.
RAPIDS_IMPORTS = ("cuml", "cudf", "dask_cudf", "cugraph", "pylibcugraph", "pylibcugraphops", "nx_cugraph", "cuspatial",
                  "cuvs", "cuxfilter", "cucim", "rmm", "pylibraft", "raft_dask", "cupy", "numba", "numba_cuda")
sh("python -c \"from importlib.metadata import packages_distributions as p; m = p(); "
   f"print(' '.join(sorted({{d for name in {RAPIDS_IMPORTS!r} for d in m.get(name, [])}})))\" > /tmp/rapids_dists.txt")
dists = open("/tmp/rapids_dists.txt").read().split()
print("removing from this session:", dists)
if dists:
    sh("pip uninstall -y -q " + " ".join(dists))
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
# every project module imports on this environment before any data or GPU time is spent
sh('PYTHONPATH=src python -c "import numpy, stokes_ft.data, stokes_ft.engine, stokes_ft.model, stokes_ft.study, '
   "stokes_ft.physics, stokes_ft.weak, stokes_ft.variants, stokes_ft.fem; print('project imports OK | numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
sh("python scripts/prepare_data.py", cwd=REPO)      # 172 MB from NGC, SHA-256 verified, 800/100/100 split with seed 0
assert tree_is_clean(), "preparing the data modified the clone"
sh("python -m pytest -q", cwd=REPO)

## Step A2: restore the previous session (`RESUME = True` only)

Finds exactly one `state-full.tar` under `/kaggle/input` (the output of the previous session, attached as a kernel source), checks it against its SHA-256 sidecar, refuses to overwrite an existing run, extracts it and verifies every checkpoint manifest. Whether the checkpoint belongs to the same configuration, dataset and architecture is checked again when training resumes.

In [ ]:
if RESUME:
    import glob
    found = sorted(glob.glob(f"/kaggle/input/**/state-{CONFIG}.tar", recursive=True))
    assert len(found) == 1, f"expected exactly one state-{CONFIG}.tar under /kaggle/input, found {found}"
    sh(f"python scripts/session_state.py unpack {found[0]} runs", cwd=REPO)
    assert tree_is_clean(), "restoring the state modified the clone"
    print("restored:", open(f"{REPO}/runs/{CONFIG}/status.json").read())
else:
    print("Step A2 skipped: RESUME is False.")


## Step B: CUDA smoke study

Six training meshes, two epochs, twenty fine-tuning iterations on two test meshes. The numbers are not results.

In [ ]:
sh("python scripts/run_study.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Step C0: device checks

On the smoke configuration: a two-epoch run paused after epoch one and resumed from its verified checkpoint, against an uninterrupted run; repeatability of two runs with the same seed; checkpoint round trip (bit-identical weights, outputs to rounding); finite gradients. The step stops the notebook if a check fails.

In [ ]:
sh("python scripts/device_checks.py --config smoke device=cuda", cwd=REPO)
assert tree_is_clean(), "the device checks modified the clone"
sh(f"cp runs/device_checks/device_checks.json {WORK}/physicsnemo-stokes-finetuning-device-checks.json", cwd=REPO)
sh("python scripts/selection_smoke.py device=cuda", cwd=REPO)   # validation-based checkpoint selection, pause and resume
assert tree_is_clean(), "the selection smoke check modified the clone"
sh(f"cp runs/selection_smoke.json {WORK}/physicsnemo-stokes-finetuning-selection-smoke.json", cwd=REPO)


## Step C: timing and memory probe

One epoch of the data-driven stage of `configs/full.yaml` (500 meshes) and 200 fine-tuning iterations on test mesh 7. The projection scales these two measurements to 500 epochs and to 10 meshes of 10000 iterations. It is an extrapolation and includes start-up cost, so it tends to overestimate.

In [ ]:
import json
sh('python scripts/run_study.py --config full name=full_probe epochs=1 pi_iters=200 "finetune.samples=[7]" '
   '"evaluation.plot_samples=[7]" training.checkpoint_every=1', cwd=REPO)
train = json.load(open(f"{REPO}/runs/full_probe/train_metrics.json"))
tune = json.load(open(f"{REPO}/runs/full_probe/finetune/finetune_7.json"))
meta = json.load(open(f"{REPO}/runs/full_probe/study_metadata.json"))["sessions"][-1]
train_h = train["seconds_per_iteration"] * 500 * 500 / 3600      # 500 epochs x 500 meshes in configs/full.yaml
tune_h = tune["seconds_per_iteration"] * 10000 * 10 / 3600       # 10000 iterations x 10 test meshes in configs/full.yaml
full_projection_h = train_h + tune_h
print(f"GPU: {meta['gpu_name']} | torch {meta['torch_version']} | CUDA {meta['cuda_version']}")
print(f"data-driven stage: {train['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {train['peak_cuda_memory_mb']:.0f} MB -> {train_h:.2f} h for 250000 iterations")
print(f"fine-tuning stage: {tune['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {tune['peak_cuda_memory_mb']:.0f} MB -> {tune_h:.2f} h for 10 meshes")
print(f"PROJECTION for step D: {full_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")
import pathlib
ckpt_mb = sum(p.stat().st_size for p in pathlib.Path(f"{REPO}/runs/full_probe/checkpoints").rglob("*") if p.is_file()) / 2**20
print(f"checkpoints kept by the probe (latest epoch only, plus one fine-tuned network): {ckpt_mb:.1f} MB")
# Preserve the probe evidence even when the full study is disabled or exceeds the budget.
for study in ("smoke_cuda", "full_probe"):
    sh(f"python scripts/results.py package runs/{study}", cwd=REPO)
    sh(f"cp runs/{study}.zip {WORK}/physicsnemo-stokes-finetuning-{study}.zip", cwd=REPO)


## Step D: full study (`configs/full.yaml`)

The MeshGraphNet checkpoint before the physics-informed stage and the ten fine-tuning networks after it are kept in `runs/full/checkpoints` and stored in the checkpoint archive. Both are evaluated on the same ten test meshes.

`CONFIG` selects the configuration (default `full`) and `STAGES` the stages (default all four). The state archive `state-<CONFIG>.tar` carries the entire run, including `checkpoints/selected` when selection is enabled. A separate results zip requires the `evaluate` stage to produce its summary.

In [ ]:
full_done = False
PACK_MARGIN_HOURS = 0.5   # left for packing the state and the results before the session ends
if RUN_FULL is not True:
    print("Step D skipped: RUN_FULL is False.")
else:
    budget = hours_left() - PACK_MARGIN_HOURS
    print(f"projected {full_projection_h:.1f} h for training and fine-tuning; {budget:.1f} h available in this session")
    print("The configuration was not reduced: if the study does not finish here it pauses at a safe point.")
    if budget < 0.25:
        print("STEP D NOT STARTED: less than 0.25 h left in this session.")
    else:
        try:
            sh(f"python scripts/run_study.py --config {CONFIG} --resume --stages {STAGES} --max-hours {budget:.2f}", cwd=REPO)
        finally:
            if os.path.isdir(f"{REPO}/runs/{CONFIG}"):
                sh(f"python scripts/session_state.py pack runs/{CONFIG} {WORK}/state-{CONFIG}.tar", cwd=REPO)
        status = json.load(open(f"{REPO}/runs/{CONFIG}/status.json"))
        print("status:", status)
        if status["state"] == "complete" and set(STAGES.split(",")) >= {"train", "predict", "finetune", "evaluate"}:
            sh(f"python scripts/plot_results.py --study runs/{CONFIG} --out runs/{CONFIG}/figures", cwd=REPO)
            sh(f"python scripts/results.py package runs/{CONFIG}", cwd=REPO)
            sh(f"python scripts/results.py bundle runs/{CONFIG}", cwd=REPO)
            sh(f"cp runs/{CONFIG}.zip {WORK}/physicsnemo-stokes-finetuning-{CONFIG}.zip && cp runs/{CONFIG}_checkpoints.zip {WORK}/physicsnemo-stokes-finetuning-{CONFIG}-checkpoints.zip", cwd=REPO)
            sh(f"cd runs/{CONFIG} && zip -q -r {WORK}/physicsnemo-stokes-finetuning-{CONFIG}-figures.zip figures", cwd=REPO)
            full_done = True
            print("STEP D COMPLETE.")
        elif status["state"] == "complete":
            if "evaluate" in status["stages_requested"]:
                sh(f"python scripts/results.py package runs/{CONFIG}", cwd=REPO)
                sh(f"cp runs/{CONFIG}.zip {WORK}/physicsnemo-stokes-finetuning-{CONFIG}.zip", cwd=REPO)
            else:
                print(f"Run outputs are in {WORK}/state-{CONFIG}.tar.")
            full_done = True
            print("STEP D COMPLETE.")
        else:
            print("STEP D PAUSED: start a new session with this version's output as a kernel source and RESUME = True.")


In [ ]:
import glob
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-stokes-finetuning-*.zip")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Steps A to C: done. Step D: {'done' if full_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")